In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd

In [3]:
subjects=["Phòng trọ","Ký túc xá tư nhân", "Căn hộ mini", "Phòng sạch sẽ"]
areas = ["gần ĐH Bách Khoa", "ngay chợ nhỏ", "đối diện KTX khu B", "khu vực an ninh Dĩ An"]
amenities = ["có máy lạnh", "wifi miễn phí", "giờ giấc tự do", "có ban công thoáng"]
prices_range = [1.5, 1.8, 2.2, 2.5, 3.0, 3.5]

In [4]:
def generate_mock_data(n=100):
    data=[]
    for _ in range(n):
        desc=f"{np.random.choice(subjects)} {np.random.choice(areas)}, {np.random.choice(amenities)}"
        price=np.random.choice(prices_range)
        area_size=np.random.randint(12,30)
        
        if price<=2.5 and area_size>=16:
            label=1
        else:
            label=1 if np.random.random()<0.2 else 0
            
        data.append([desc,price,area_size,label])
    return pd.DataFrame(data,columns=["Description","Price","Area","Label"])

In [5]:
df = generate_mock_data(100)
print("Dữ liệu mẫu:")
print(df.head())

Dữ liệu mẫu:
                                         Description  Price  Area  Label
0  Ký túc xá tư nhân khu vực an ninh Dĩ An, có má...    2.5    25      1
1        Phòng sạch sẽ gần ĐH Bách Khoa, có máy lạnh    2.2    20      1
2  Căn hộ mini khu vực an ninh Dĩ An, giờ giấc tự do    2.5    29      1
3     Phòng sạch sẽ gần ĐH Bách Khoa, giờ giấc tự do    1.5    20      1
4     Phòng trọ khu vực an ninh Dĩ An, wifi miễn phí    3.5    18      0


In [6]:
df.to_csv('../data/raw/mock_data.csv', index=False)

In [7]:
import tensorflow as tf

In [8]:
y=df.pop("Label")

X={
    "Description":df["Description"].values,
    "Price":df["Price"].values,
    "Area":df["Area"].values
}

print("Cấu trúc X:", X["Description"][:2])

Cấu trúc X: ['Ký túc xá tư nhân khu vực an ninh Dĩ An, có máy lạnh'
 'Phòng sạch sẽ gần ĐH Bách Khoa, có máy lạnh']


In [9]:
raw_dataset=tf.data.Dataset.from_tensor_slices((X,y))

BATCH_SIZE=32

dataset=(
    raw_dataset
    .shuffle(buffer_size=len(df))
    .batch(batch_size=BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [10]:
for features,labels in dataset.take(1):
    print("1. Hình dáng (Shape) của nhãn:", labels.shape)
    
    print("\n2. Dữ liệu cột 'price' (5 dòng đầu):")
    print(features['Price'][:5].numpy())
    
    print("\n3. Dữ liệu cột 'description' (2 dòng đầu):")
    for desc in features['Description'][:2]:
        print("-", desc.numpy().decode('utf-8'))

1. Hình dáng (Shape) của nhãn: (32,)

2. Dữ liệu cột 'price' (5 dòng đầu):
[3.  1.8 2.5 2.5 3. ]

3. Dữ liệu cột 'description' (2 dòng đầu):
- Căn hộ mini đối diện KTX khu B, có ban công thoáng
- Căn hộ mini đối diện KTX khu B, wifi miễn phí


In [11]:
from tensorflow.keras import layers

In [12]:
vectorize_layers=layers.TextVectorization(
    max_tokens=1000,
    output_mode='int',
    output_sequence_length=15
)

vectorize_layers.adapt(df["Description"].values)

In [13]:
price_norm=layers.Normalization(axis=None)
price_norm.adapt(df["Price"].values)

In [14]:
area_norm=layers.Normalization(axis=None)
area_norm.adapt(df["Area"].values)

In [15]:
input_desc=layers.Input(shape=(1,),dtype="string",name="description")
input_area=layers.Input(shape=(1,),name="area")
input_price=layers.Input(shape=(1,),name="price")

In [16]:
x1=vectorize_layers(input_desc)
x1=layers.Embedding(input_dim=1000,output_dim=16)(x1)
x1=layers.GlobalAveragePooling1D()(x1)

In [17]:
x2=price_norm(input_price)
x3=area_norm(input_area)

In [18]:
combined=layers.Concatenate()([x1,x2,x3])

In [19]:
z=layers.Dense(units=16,activation="relu")(combined)
z=layers.Dense(units=8,activation="relu")(z)
output=layers.Dense(units=1,activation="sigmoid",name="label")(z)

model=tf.keras.Model(
    inputs=[input_desc,input_price,input_area],
    outputs=output
)

In [20]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ description         │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text_vectorization  │ (None, 15)        │          0 │ description[0][0] │
│ (TextVectorization) │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 15, 16)    │     16,000 │ text_vectorizati… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ price (InputLayer)  │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ area (InputLayer)   │ (None, 1)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 16)        │          0 │ embedding[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 1)         │          3 │ price[0][0]       │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 1)         │          3 │ area[0][0]        │
│ (Normalization)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 18)        │          0 │ global_average_p… │
│ (Concatenate)       │                   │            │ normalization[0]… │
│                     │                   │            │ normalization_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 16)        │        304 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 8)         │        136 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ label (Dense)       │ (None, 1)         │          9 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 16,455 (64.29 KB)

 Trainable params: 16,449 (64.25 KB)

 Non-trainable params: 6 (32.00 B)